# Practice 5. Tools, workflows and agents

Help a quality engineer investigate a change in the defect rate on a production line.
Use the same three tools in a workflow and an agent. Inspect how each system chooses its steps.

Approx. from 60 to 90 minutes. The main path is complete and needs no code changes.
Choose a scenario, predict what happens, run it, then inspect the implementation.

You need basic Python and pandas. Framework terms are explained where they first appear.

Your starting question is "Did quality get worse on plant's production line L-2 on November 9?"
The useful result is a short report with counts, product comparisons and a next check.

You will finish with three checkpoint answers and a short architecture choice. Use it in your course project

## Before you start

Open [the setup instructions](../README.md) and install the environment once.
Select the practice environment as the kernel. Click a code cell and press Shift+Enter to run it.


Live is the default and uses the course model. Set the course URL and token in `.env`
as described in the README. Model responses can differ between runs.

Set `PRACTICE_MODE=replay` before starting Jupyter for the supplied offline examples.
Replay needs no keys and makes no model requests. Tools and graphs still execute
locally. Only model responses come from recordings.

Work one section at a time. Read its prediction question before running the example.
You can also use Run All to check setup, then return to each experiment.

The buttons run prepared examples. The step viewer inspects a completed execution,
not a paused model. Expand raw payloads only when you need the exact fields.

For each written checkpoint, double-click the "Your answer" cell below it, type your
answer and press Shift+Enter. Save a personal copy of the notebook to keep your notes.
The radio-button checks give feedback but do not grade or save a score.

In [2]:
import os
import pandas as pd
from IPython.display import display

from quality_assistant.tools import DATA, TOOLS, get_quality_summary, compare_periods
from quality_assistant.models import make_model
from quality_assistant.systems import report_input
from quality_assistant.ui import source_view, checkpoint, explorer, step_view, rates_view, graph_view

MODE = os.environ.get("PRACTICE_MODE", "live")
print("Model mode", MODE)
# These two CSV files are the entire data source for the tools.
batches = pd.read_csv(DATA / "batches.csv")
events = pd.read_csv(DATA / "events.csv")
display(batches)
display(events)

Model mode replay


,batch_id,date,line_id,product_id,produced,defects
0,B-001,2026-11-08,L-2,A,100,1
1,B-002,2026-11-08,L-2,A,900,9
2,B-003,2026-11-08,L-2,B,1000,40
3,B-004,2026-11-08,L-2,B,1000,40
4,B-005,2026-11-09,L-2,A,100,10
5,B-006,2026-11-09,L-2,A,1900,50
6,B-007,2026-11-09,L-2,B,500,40
7,B-008,2026-11-09,L-2,B,500,40
8,B-009,2026-11-08,L-1,A,1000,20
9,B-010,2026-11-09,L-1,A,1000,20


,event_id,date,line_id,event
0,E-001,2026-11-08,L-2,Routine inspection. No fault recorded.
1,E-002,2026-11-09,L-2,Sealing unit adjusted before production.
2,E-003,2026-11-09,L-2,New film roll introduced. Supplier unchanged.
3,E-004,2026-11-08,L-1,Routine inspection. No fault recorded.


## 1. Start with the data

A batch records produced items and defective items. An event records work on a line.
L-2 has a change in quality. L-1 is stable. L-3 has no batch records.
Dates are inclusive and fixed to November 8 and 9, 2026.

A defect rate is total defects divided by total produced items.
A mean of batch percentages gives small and large batches equal weight.
That can answer the wrong question.

Predict which calculation answers "What share of all items was defective?"
Then run the cell and compare the two numbers. The result dictionary keeps counts
and batch IDs so you can check the answer against the rows above.

In [3]:
current = get_quality_summary.invoke(dict(line_id="L-2", start="2026-11-09", end="2026-11-09"))
selected = batches[(batches.line_id == "L-2") & (batches.date == "2026-11-09")]
naive_rate = (100 * selected.defects / selected.produced).mean()
print("Mean of batch percentages", round(naive_rate, 4))
print("Weighted rate", current["defect_rate_pct"])
print("Defects and produced items", current["defects"], current["produced"])
comparison = compare_periods.invoke(report_input())
rates_view(comparison)

Mean of batch percentages 7.1579
Weighted rate 4.6667
Defects and produced items 140 3000


Product,Period,Defect rate
A,before,1.0%
A,after,3.0%
B,before,4.0%
B,after,8.0%


Expected in both modes. The weighted rate is 4.6667 percent, from 140 defective items
out of 3000. Product A rises from 1 to 3 percent. Product B rises from 4 to 8 percent.

The calculations are plain pandas. No model is trained or needed for them. A tool exposes a useful operation to the language model.

Open `get_quality_summary`, then `select_batches` and `totals` below.
`@tool` uses the function name, docstring and type hints to describe the tool.
`.func` exposes the original Python function for inspection.
`select_batches` validates and filters. `totals` divides summed counts.

In [4]:
from quality_assistant.tools import select_batches, totals

display(source_view(get_quality_summary.func, select_batches, totals, compare_periods.func))

Accordion(children=(HTML(value='<pre>@tool\ndef get_quality_summary(line_id: str, start: str, end: str) -&gt; ...

## 2. Follow one tool call

The model receives a tool name, description and argument schema.
It may return a structured tool call instead of a final answer.
The application validates and executes the function. A tool result goes back to the model.

First predict the tool and arguments for this question.
What was the defect rate for L-2 on November 9?

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from quality_assistant.systems import SYSTEM
from quality_assistant.tools import pretty

model = make_model(MODE)
messages = [SystemMessage(content=SYSTEM),
            HumanMessage(content="What was the defect rate for L-2 on 2026-11-09?")]

# bind_tools gives the model a schema. It's not the function's execution
bound = model.bind_tools([get_quality_summary])
print(pretty(get_quality_summary.args))

{
  "line_id": {
    "title": "Line Id",
    "type": "string"
  },
  "start": {
    "title": "Start",
    "type": "string"
  },
  "end": {
    "title": "End",
    "type": "string"
  }
}


`SystemMessage` sets the task rules. `HumanMessage` holds the question.
`bind_tools` supplies tool descriptions to the model.

The next call returns an `AIMessage`. Inspect its `tool_calls` field for the function
name, arguments and call ID. A proposal is not yet a computed result.

In [6]:
proposal = bound.invoke(messages)
print(pretty(proposal.tool_calls))
assert proposal.tool_calls, "The course profile must support native tool calling"

[
  {
    "name": "get_quality_summary",
    "args": {
      "line_id": "L-2",
      "start": "2026-11-09",
      "end": "2026-11-09"
    },
    "id": "chatcmpl-tool-8ad150e9eac45923",
    "type": "tool_call"
  }
]


Now the application executes the requested function. A `ToolMessage` sends its result
back with the same call ID. The final model call can use this observation to answer.

Follow the four variables below. `call` is the request, `observation` is the computed
result, `tool_message` is its message wrapper, and `final` is the model's explanation.

In [7]:
call = proposal.tool_calls[0]
assert call["name"] == get_quality_summary.name

observation = get_quality_summary.invoke(call["args"])
tool_message = ToolMessage(content=pretty(observation), tool_call_id=call["id"], name=call["name"])
print("Tool result")
print(tool_message.content)

final = bound.invoke([*messages, proposal, tool_message])
print("Final answer")
print(final.content)

Tool result
{
  "status": "ok",
  "line_id": "L-2",
  "start": "2026-11-09",
  "end": "2026-11-09",
  "produced": 3000,
  "defects": 140,
  "defect_rate_pct": 4.6667,
  "products": [
    {
      "product_id": "A",
      "produced": 2000,
      "defects": 60,
      "defect_rate_pct": 3.0
    },
    {
      "product_id": "B",
      "produced": 1000,
      "defects": 80,
      "defect_rate_pct": 8.0
    }
  ],
  "batch_ids": [
    "B-005",
    "B-006",
    "B-007",
    "B-008"
  ]
}
Final answer
On 2026-11-09, line L-2 had a weighted defect rate of 4.67 percent (140 defects out of 3,000 units).

By product:
- Product A: 3.0 percent (60 of 2,000)
- Product B: 8.0 percent (80 of 1,000)

Source: get_quality_summary for L-2, 2026-11-09 to 2026-11-09, batches B-005 through B-008.

Product B is notably higher than A. A practical next check would be to review the most recent maintenance or process change on the B line before that date to see if anything shifted.


A call ID links the result to the proposal.
Here the function checks the line and dates.

In the next control, run "No batches" and "Unknown line". Predict which should
return an unknown rate and which should reject the input. A valid empty query is
different from an invalid query. Both are handled locally, without a model call.

The recorded final answer says "B line". B is a product on L-2, not a separate line.
The tool result is correct, but the explanation still needs a check.

In [8]:
from quality_assistant.ui import tool_lab

display(tool_lab())

In [9]:
display(checkpoint(
    "Which component calculates the defect rate?",
    ["The language model", "The Python tool", "The tool description"],
    "The Python tool",
    "The model selects an operation. The tool calculates it from source records."))

Checkpoint 1. In your own words, explain why this tool is useful.
Name one input check and one part of the result that lets you verify the number.

**Your answer 1**

Replace this sentence with two or three sentences about the calculation, an input
check and the source records you would inspect.

<details><summary>One possible answer, after your attempt</summary>

The Python tool calculates a weighted rate from the full dataset, rather than asking
the model to estimate it. It rejects an unknown line. I would check the produced and
defect counts for B-005 to B-008, which support 140 defects out of 3000 items.

</details>

## 3. A workflow for a regular report

A daily report has a known process. Read current statistics, compare the same products,
then inspect events if a product rose by at least 1 percentage point.
Finally ask the model to explain the collected facts.

The threshold is an illustrative reporting rule.
Code chooses these steps, the language model writes the explanation, LangGraph stores results in state and selects a conditional edge.

A node is a Python step. State is the dictionary of inputs and results carried between nodes. An edge decides which step follows. A conditional edge selects a branch.

Read the graph first. Predict whether the events node is needed on a stable line.
Then run the default L-2 example and inspect the comparison step.

In [10]:
from quality_assistant.systems import run_workflow, build_workflow, needs_events

workflow_model = make_model(MODE)
display(graph_view(build_workflow(make_model(MODE))))
workflow = run_workflow(workflow_model, "L-2")
print("Completed nodes", [s["node"] for s in workflow["steps"]])
print("Report preview", workflow["answer"].splitlines()[0])
display(step_view(workflow))

Completed nodes ['summary', 'comparison', 'events', 'report']
Report preview Line L-2 report for 2026-11-09.


In [11]:
display(explorer("workflow", MODE))

Expected paths. L-2 includes the events node. L-1 skips it.
L-3 ends with an incomplete report because no batches are available.
A missing rate is unknown, not zero.

In the explorer, select Stable L-1 and click Run. Check that events is skipped.
Then select No data on L-3 and run again. Inspect why the report is incomplete.

The slider and Previous/Next buttons inspect a completed run. The raw update panel
shows only what a node returned. Full state includes earlier results too.

### Read the workflow code

Open `build_workflow` first. `StateGraph(ReportState)` declares the state fields.
`add_node` registers a function. Each function returns only the fields it changes.
LangGraph merges that update into state.

`add_edge` fixes the next step. `add_conditional_edges` calls `needs_events` to select
one of its named paths. `compile` makes the graph runnable. `run_workflow` collects
node updates with `stream`, which is why the viewer can show intermediate state.

The helper functions below are the actual tool calls inside the nodes.

In [12]:
from quality_assistant.systems import ReportState, read_summary, read_comparison, read_events

display(source_view(build_workflow, ReportState, needs_events,
                    read_summary, read_comparison, read_events, run_workflow))

Accordion(children=(HTML(value='<pre>def build_workflow(model):\n    def report(state):\n        if state[&#x2...

In [13]:
stable = run_workflow(make_model(MODE), "L-1")
missing = run_workflow(make_model(MODE), "L-3")
assert "events" not in [s["node"] for s in stable["steps"]]
assert "cannot be completed" in missing["answer"]
print("PASS. Stable data skips events. Missing data is not reported as normal.")

PASS. Stable data skips events. Missing data is not reported as normal.


## 4. An agent for different questions

An engineer may ask for a product comparison, an event list or a regular report.
These questions need different tools. A fixed report can do unnecessary work.

In the agent loop, the model selects tools after observing earlier results.
LangGraph executes them and returns their results to the model.
The loop ends with a final answer or a turn limit.

This is a ReAct-style loop with native tool calls.
Inspect its actions and observations. Hidden reasoning is not needed to follow execution.
The agent is general within these three tools and datasets. It has no other access.

The prompt still gives the regular-report rule. The agent does not discover company policy by itself. Its extra freedom is choosing calls for different questions.

In [14]:
from quality_assistant.systems import SCENARIOS, run_agent, build_agent

agent_model = make_model(MODE)
display(graph_view(build_agent(make_model(MODE)), "agent"))
agent = run_agent(agent_model, "regular_report")
print(SCENARIOS["regular_report"])
display(step_view(agent))

Prepare a regular report for L-2. Before 2026-11-08 to 2026-11-08. After 2026-11-09 to 2026-11-09. Include the current summary, compare products and inspect relevant events if needed. Separate facts from hypotheses.


In [15]:
display(explorer("agent", MODE))

In the explorer, choose Read events only. Predict whether quality statistics are needed,
then run and inspect the calls. Next try Compare products on L-2.

Set the turn limit to 1 and run again. One turn is one model invocation, not one tool call.
A model can propose several calls in a turn. Compare proposed and executed counts.
At the limit, pending calls are blocked and the run is marked incomplete.

### Read the agent code

`bind_tools(TOOLS)` offers all three tools. The model node appends a response.
`ToolNode` executes proposed calls and appends their observations. The next model call
receives that longer conversation.

`add_messages` in `AgentState` keeps earlier messages instead of replacing the history.
`route` checks for tool calls, a final answer or an exhausted turn budget.
Open the state and builder to find these parts.

LangChain also offers `create_agent` for a shorter setup. It supplies a standard agent
loop. The explicit graph above is useful for understanding and changing the control.

In [16]:
from quality_assistant.systems import AgentState

display(source_view(AgentState, build_agent))

Accordion(children=(HTML(value='<pre>class AgentState(TypedDict, total=False):\n    messages: Annotated[list, ...

In [17]:
from quality_assistant.systems import build_short_agent

short_result = build_short_agent(make_model(MODE)).invoke(
    {"messages": [HumanMessage(content=SCENARIOS["events_only"])]},
    {"recursion_limit": 16})
print(short_result["messages"][-1].content)
display(source_view(build_short_agent))

L-2 maintenance and changeover events, 2026-11-08 to 2026-11-09:

- E-001, 2026-11-08: Routine inspection. No fault recorded.
- E-002, 2026-11-09: Sealing unit adjusted before production.
- E-003, 2026-11-09: New film roll introduced. Supplier unchanged.

Source: get_maintenance_events for L-2. No defect statistics included, as requested.


Accordion(children=(HTML(value='<pre>def build_short_agent(model):\n    return create_agent(model=model, tools...

In [18]:
display(checkpoint(
    "A fixed daily report already works. What justifies trying an agent?",
    ["It has more model calls", "Questions need different paths through the tools", "It always gives better reports"],
    "Questions need different paths through the tools",
    "Dynamic selection can help with varied questions. A fixed process can still be best for a repeated report."))

Checkpoint 2. Choose one scenario. Record its tool sequence and explain why those calls fit the question.
Would the regular-report workflow do unnecessary work for that question?

**Your answer 2**

Name your scenario and the tools that executed. Explain why one call was needed or
unnecessary. Say what changed when you reduced the turn limit.

<details><summary>One possible answer, after your attempt</summary>

For Read events only, the agent executes get_maintenance_events and then answers.
The regular report would also calculate rates and compare periods, which the question
does not need. With a one-turn limit the model can propose a call, but the application
stops before executing it. The investigation is incomplete.

</details>

## 5. Compare the same task

Compare the regular report on L-2 with the same tools and data.
The workflow and agent differ in who selects the steps.
Different questions demonstrate flexibility, not a fair quality comparison.

Read both reports side by side below. Check the overall rate, product changes and sources.
Then inspect resource counts. `tool_calls` counts executed calls, not proposals.
`model_calls` counts model invocations. Token counts measure input and output size.

The recordings are one run per system. They do not establish a statistical quality gain.
Do the reports distinguish the observed change from possible explanations?

In [19]:
from quality_assistant.systems import metrics
from quality_assistant.evaluation import check_result

from quality_assistant.ui import report_comparison

display(report_comparison(workflow, agent))
checks = pd.DataFrame([check_result(workflow, "workflow"), check_result(agent, "agent")])
resources = pd.DataFrame([metrics(workflow, workflow_model, "workflow"),
                          metrics(agent, agent_model, "agent")])
display(checks)
display(resources)
print("Resource counts describe this run only. They are not a benchmark.")
print("Recorded model seconds sum the original API calls. They are not graph wall time.")
display(source_view(check_result, metrics))

Workflow report,Agent report
"Line L-2 report for 2026-11-09.After-period summary (B-005 to B-008): 140 defects in 3000 units, a 4.67 percent rate. Product A was 3.0 percent (60/2000). Product B was 8.0 percent (80/1000).Comparison to 2026-11-08:Product A rose 2.0 points from 1.0 percent to 3.0 percent.Product B rose 4.0 points from 4.0 percent to 8.0 percent.Events on 2026-11-09 include a sealing unit adjustment (E-002) and a new film roll (E-003). These coincide with the rate increases but do not prove causation.Next check: Inspect the sealing unit settings and film roll adhesion on Product B samples from batch B-008.","L-2 report, 2026-11-09.Facts (get_quality_summary, batches B-005 to B-008): 3000 units, 140 defects, 4.67 percent. Product A: 2000 units, 60 defects, 3.0 percent. Product B: 1000 units, 80 defects, 8.0 percent.Comparison (compare_periods, 2026-11-08 to 2026-11-09): Product A rose 2.0 points (1.0 to 3.0). Product B rose 4.0 points (4.0 to 8.0). Both matched products increased.Events (get_maintenance_events): E-002 sealing unit adjusted before production. E-003 new film roll introduced, supplier unchanged.Hypotheses: The sealing adjustment or new film roll may relate to the rises, but timing alone does not prove causation.Next check: Inspect seal quality and film adhesion on Product B samples from B-005 to B-008."


,architecture,matched_rates_ok,read_events,mentions_products,limit_words_present,completed
0,workflow,True,True,True,True,True
1,agent,True,True,True,True,True


,architecture,tool_calls,model_calls,input_tokens,output_tokens,recorded_model_seconds
0,workflow,3,1,915,229,3.144
1,agent,3,3,4156,516,4.511


Resource counts describe this run only. They are not a benchmark.
Recorded model seconds sum the original API calls. They are not graph wall time.


Accordion(children=(HTML(value='<pre>def check_result(result, architecture):\n    observations = tool_results(...

The checks validate structured tool results and simple report coverage.
`matched_rates_ok` checks the numbers returned by tools, not the model's arithmetic.
`limit_words_present` only finds words about uncertainty. It cannot tell whether the
report actually avoids a causal claim. Open the check implementation if you want to
see exactly what a True value means.


They are deliberately limited. A report can mention the right numbers and still make a bad causal claim.
For example, the recorded agent suggests Product B samples from B-005 to B-008.
Check the input rows. B-005 and B-006 are Product A, so B samples are B-007 and B-008.
This wording error remains visible even when the structured calculations pass.
Read the final text and inspect the evidence before accepting it.

For a repeated report, a workflow makes required checks explicit.
For varied questions, an agent can choose a smaller or different set of operations.
A useful production design can combine both.

In [20]:
display(checkpoint(
    "A maintenance event and a higher defect rate occur on the same day. What follows?",
    ["Maintenance caused the defects", "The event is a hypothesis to investigate", "The rate calculation is invalid"],
    "The event is a hypothesis to investigate",
    "Check process conditions and other changes. Timing alone does not establish causation."))

Checkpoint 3. Choose workflow or agent for your project task.
Give one reason, one expected failure and one test that would change your choice.

**Your answer 3**

Describe your task, choose an architecture and justify it. Give one failure case and
one observable check. You can use the daily report if you do not have a project yet.

<details><summary>One possible answer, after your attempt</summary>

I would use a workflow for the fixed daily quality report. It makes product matching
and the events threshold explicit. Missing batches could otherwise be mistaken for a
zero rate. I would test that an empty period returns an incomplete report with no rate.
For varied follow-up questions, I would compare this with a bounded agent.

</details>

## 6. Check yourself and explore

The next cell checks the supplied data, graph branches and prepared replay scenarios.
A passing check verifies the examples, not your written answers or every report claim.
Live responses may differ.
Keep your three checkpoint answers and a short conclusion with the completed notebook.

Success means that you can explain one tool exchange, follow a workflow branch,
and justify when dynamic tool selection helps. No new implementation is required.

In [21]:
from quality_assistant.evaluation import verify_reference

if MODE == "replay":
    verify_reference()
    print("PASS. Reference facts, graph branches and prepared agent scenarios checked.")
else:
    print("Live mode. Inspect the actual trace and report, then run the replay reference separately.")

PASS. Reference facts, graph branches and prepared agent scenarios checked.


### Small experiments

Try the missing-data scenario. Find the tool result that prevents a numerical answer.
Reduce the agent turn limit. Find the difference between a proposed and an executed call.
Open a tool implementation and trace each returned field back to the data.

Changing a prompt or tool schema changes the model request.
Replay accepts exact recorded requests only. It raises a clear error for a new one.
Use live mode for free-form questions and code changes that affect requests.

In [22]:
# This experiment is optional and only makes a request in live mode.
custom_question = "Compare product A on L-2 on 2026-11-08 and 2026-11-09."
if MODE == "live":
    print(run_agent(make_model(MODE), custom_question)["answer"])
else:
    print("Prepared scenarios work offline. Set PRACTICE_MODE=live for a new question.")

Prepared scenarios work offline. Set PRACTICE_MODE=live for a new question.


### Optional extension

Add one read-only tool based on a known calculation or library.
For example, return the batch with the largest number of defective items.
Describe its inputs, errors and output. Give three questions where it is useful,
unnecessary or lacks data. Connect it to the agent in live mode.

For a project, the same pattern can wrap an existing ML model.
Training that model is separate from this practice.

### Common problems

If imports fail, select the practice kernel and install the locked environment.
If a name is undefined, run the earlier setup cells in order.
If controls do not respond, check that the kernel is running. Widgets need a live kernel,
even in replay. A static preview can display saved output but cannot execute buttons.


If a widget is unavailable, use the normal examples and the executed reference notebook.
A replay miss means the question, schema or earlier observation differs from the recording.
Restore the supplied version or use live mode. Do not substitute a similar saved answer.
A live connection error needs a check of the course URL, token and network access.
Never paste the token into the notebook or a log.

### Sources

LangChain, Tools and Models, documentation checked in 2026.
https://docs.langchain.com/oss/python/langchain/tools
https://docs.langchain.com/oss/python/langchain/models

LangGraph, Workflows and agents, documentation checked in 2026.
https://docs.langchain.com/oss/python/langgraph/workflows-agents

LangChain, Agents, documentation checked in 2026.
https://docs.langchain.com/oss/python/langchain/agents